In [ ]:
import pygame
import socket
import requests
import time
import math
import os
import csv
from datetime import datetime

# ==========================
# CONFIG
# ==========================
ROVER_IP = "192.168.4.1"
ROVER_PORT = 4210

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

def send_heading(deg, duration_ms):
    msg = f"HEAD,{deg},{duration_ms}"
    sock.sendto(msg.encode(), (ROVER_IP, ROVER_PORT))

def send_stop():
    msg = "0,0,0,0"
    sock.sendto(msg.encode(), (ROVER_IP, ROVER_PORT))

def get_status():
    try:
        r = requests.get(f"http://{ROVER_IP}/status", timeout=0.2)
        return r.json()
    except Exception as e:
        print("Status error:", e)
        return None

# ==========================
# PYGAME SETUP
# ==========================
pygame.init()
pygame.font.init()

WIDTH, HEIGHT = 900, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Rover Control Station")

font = pygame.font.Font(None, 24)
bigfont = pygame.font.Font(None, 32)

clock = pygame.time.Clock()

# ==========================
# UI ELEMENTS
# ==========================
# Telemetry panel
TELEM_X, TELEM_Y = 20, 20
TELEM_W, TELEM_H = 400, 260
telemetry_rect = pygame.Rect(TELEM_X, TELEM_Y, TELEM_W, TELEM_H)

# Compass
COMPASS_CX = 650
COMPASS_CY = 250
COMPASS_R = 150

# Input fields
heading_box = pygame.Rect(20, HEIGHT - 80, 150, 35)
duration_box = pygame.Rect(190, HEIGHT - 80, 150, 35)
heading_text = ""
duration_text = ""

active_heading = False
active_duration = False

# Arrow buttons (small beside each input)
heading_up = pygame.Rect(heading_box.right + 5, heading_box.y, 25, 17)
heading_down = pygame.Rect(heading_box.right + 5, heading_box.y + 18, 25, 17)

duration_up = pygame.Rect(duration_box.right + 5, duration_box.y, 25, 17)
duration_down = pygame.Rect(duration_box.right + 5, duration_box.y + 18, 25, 17)

# Buttons
send_button = pygame.Rect(360, HEIGHT - 80, 80, 35)
stop_button = pygame.Rect(460, HEIGHT - 80, 80, 35)

# Command heading
command_heading = 0.0

# Compass drag
compass_dragging = False

# Telemetry / status
last_status_time = 0
status_cache = None
time_since_last_comm = 0.0
last_comm_timestamp = None

# Heading averaging
heading_samples = []
avg_heading = None
current_cmd_heading = None
current_cmd_duration = None

# Logging
logs_dir = "logs"
os.makedirs(logs_dir, exist_ok=True)
logging_active = False
current_log_file = None
current_log_writer = None

# Blinking cursor
cursor_visible = True
cursor_timer = 0
CURSOR_BLINK_MS = 500

# ==========================
# HELPER FUNCTIONS
# ==========================
def angle_wrap_deg(a):
    while a < 0:
        a += 360
    while a >= 360:
        a -= 360
    return a

def add_heading_sample(h):
    global heading_samples, avg_heading
    heading_samples.append(h)
    if heading_samples:
        # simple circular mean
        s = sum(math.sin(math.radians(x)) for x in heading_samples)
        c = sum(math.cos(math.radians(x)) for x in heading_samples)
        avg = math.degrees(math.atan2(s, c))
        avg_heading = angle_wrap_deg(avg)
    else:
        avg_heading = None

def draw_telemetry_panel():
    pygame.draw.rect(screen, (230, 230, 230), telemetry_rect)
    pygame.draw.rect(screen, (0, 0, 0), telemetry_rect, 2)

    lines = []

    if status_cache:
        lines.append(f"Actual heading: {status_cache['heading']:.2f}°")
    else:
        lines.append("Actual heading: ---")

    if avg_heading is not None:
        lines.append(f"Average heading: {avg_heading:.2f}°")
    else:
        lines.append("Average heading: ---")

    if current_cmd_heading is not None and current_cmd_duration is not None:
        lines.append(f"Command: {current_cmd_heading:.1f}° for {current_cmd_duration:.1f}s")
    else:
        lines.append("Command: ---")

    if status_cache:
        lines.append(f"GPS: {status_cache['lat']:.6f}, {status_cache['lon']:.6f}")
        lines.append(f"GPS fix: {'YES' if status_cache['fix'] else 'NO'}")
    else:
        lines.append("GPS: ---")
        lines.append("GPS fix: ---")

    # time since last comm
    lines.append(f"Time since last comm: {time_since_last_comm:.2f}s")

    # active / idle
    if status_cache and status_cache.get("active", 0) == 1:
        lines.append("Robot status: ACTIVE")
    else:
        lines.append("Robot status: IDLE")

    y = TELEM_Y + 8
    for line in lines:
        surf = font.render(line, True, (0, 0, 0))
        screen.blit(surf, (TELEM_X + 8, y))
        y += 22

def draw_compass(heading_actual, heading_cmd):
    pygame.draw.circle(screen, (245, 245, 245), (COMPASS_CX, COMPASS_CY), COMPASS_R)
    pygame.draw.circle(screen, (0, 0, 0), (COMPASS_CX, COMPASS_CY), COMPASS_R, 2)

    # tick marks every 30°
    for deg in range(0, 360, 30):
        rad = math.radians(deg - 90)
        x1 = COMPASS_CX + math.cos(rad) * (COMPASS_R - 10)
        y1 = COMPASS_CY + math.sin(rad) * (COMPASS_R - 10)
        x2 = COMPASS_CX + math.cos(rad) * (COMPASS_R - 2)
        y2 = COMPASS_CY + math.sin(rad) * (COMPASS_R - 2)
        pygame.draw.line(screen, (0, 0, 0), (x1, y1), (x2, y2), 2)

    # cardinal labels
    for label, deg in [("N", 0), ("E", 90), ("S", 180), ("W", 270)]:
        rad = math.radians(deg - 90)
        x = COMPASS_CX + math.cos(rad) * (COMPASS_R - 25)
        y = COMPASS_CY + math.sin(rad) * (COMPASS_R - 25)
        surf = bigfont.render(label, True, (0, 0, 0))
        rect = surf.get_rect(center=(x, y))
        screen.blit(surf, rect)

    # actual heading pointer (red)
    if heading_actual is not None:
        rad_act = math.radians(heading_actual - 90)
        x_act = COMPASS_CX + math.cos(rad_act) * (COMPASS_R - 35)
        y_act = COMPASS_CY + math.sin(rad_act) * (COMPASS_R - 35)
        pygame.draw.line(screen, (200, 0, 0), (COMPASS_CX, COMPASS_CY), (x_act, y_act), 4)

    # command heading pointer (blue)
    rad_cmd = math.radians(heading_cmd - 90)
    x_cmd = COMPASS_CX + math.cos(rad_cmd) * (COMPASS_R - 60)
    y_cmd = COMPASS_CY + math.sin(rad_cmd) * (COMPASS_R - 60)
    pygame.draw.line(screen, (0, 0, 200), (COMPASS_CX, COMPASS_CY), (x_cmd, y_cmd), 4)

    label = font.render(
        f"Actual: {heading_actual:6.2f}°" if heading_actual is not None else "Actual: ---",
        True, (0, 0, 0)
    )
    screen.blit(label, (COMPASS_CX - 80, COMPASS_CY + COMPASS_R + 10))

    label2 = font.render(f"Command: {heading_cmd:6.2f}°", True, (0, 0, 0))
    screen.blit(label2, (COMPASS_CX - 80, COMPASS_CY + COMPASS_R + 30))

def compass_angle_from_mouse(mx, my):
    dx = mx - COMPASS_CX
    dy = my - COMPASS_CY
    angle = math.degrees(math.atan2(dy, dx)) + 90  # 0 at north
    return angle_wrap_deg(angle)

def start_logging_session():
    global logging_active, current_log_file, current_log_writer
    if logging_active:
        return
    ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = os.path.join(logs_dir, f"log_{ts}.csv")
    current_log_file = open(filename, "w", newline="")
    current_log_writer = csv.writer(current_log_file)
    current_log_writer.writerow(["timestamp", "cmd_heading", "actual_heading", "lat", "lon"])
    logging_active = True
    print(f"[LOG] Started logging to {filename}")

def stop_logging_session():
    global logging_active, current_log_file, current_log_writer, heading_samples, avg_heading
    if logging_active and current_log_file:
        current_log_file.close()
        print("[LOG] Stopped logging")
    logging_active = False
    current_log_file = None
    current_log_writer = None
    heading_samples = []
    avg_heading = None

def log_status_row(status):
    if not logging_active or current_log_writer is None:
        return
    ts = datetime.now().isoformat()
    ah = status["heading"]
    lat = status["lat"]
    lon = status["lon"]
    ch = current_cmd_heading if current_cmd_heading is not None else ""
    current_log_writer.writerow([ts, ch, ah, lat, lon])

# ==========================
# MAIN LOOP
# ==========================
running = True
display_heading_actual = None

while running:
    dt = clock.tick(30)
    screen.fill((240, 240, 240))

    # Cursor blink
    cursor_timer += dt
    if cursor_timer >= CURSOR_BLINK_MS:
        cursor_timer = 0
        cursor_visible = not cursor_visible

    # STATUS UPDATE (event-driven logging)
    status = get_status()
    now = time.time()
    if status:
        status_cache = status
        last_status_time = now
        last_comm_timestamp = now
        time_since_last_comm = 0.0

        # print full raw telemetry to terminal
        print(status)

        # handle active/idle transitions for logging
        active = status.get("active", 0) == 1
        if active and not logging_active:
            start_logging_session()
        elif not active and logging_active:
            stop_logging_session()

        # if active, log and update averaging
        if active:
            log_status_row(status)
            add_heading_sample(status["heading"])

        display_heading_actual = status["heading"]
    else:
        if last_comm_timestamp is not None:
            time_since_last_comm = now - last_comm_timestamp

    # DRAW TELEMETRY
    draw_telemetry_panel()

    # DRAW COMPASS
    draw_compass(display_heading_actual, command_heading)

    # INPUT LABELS
    heading_label = font.render("Heading (deg):", True, (0, 0, 0))
    duration_label = font.render("Duration (sec):", True, (0, 0, 0))
    screen.blit(heading_label, (heading_box.x, heading_box.y - 20))
    screen.blit(duration_label, (duration_box.x, duration_box.y - 20))

    # INPUT BOXES
    pygame.draw.rect(screen, (255, 255, 255), heading_box)
    pygame.draw.rect(screen, (0, 0, 0), heading_box, 2)
    pygame.draw.rect(screen, (255, 255, 255), duration_box)
    pygame.draw.rect(screen, (0, 0, 0), duration_box, 2)

    heading_surf = font.render(heading_text, True, (0, 0, 0))
    duration_surf = font.render(duration_text, True, (0, 0, 0))
    screen.blit(heading_surf, (heading_box.x + 5, heading_box.y + 8))
    screen.blit(duration_surf, (duration_box.x + 5, duration_box.y + 8))

    # Blinking cursor
    if active_heading:
        cx = heading_box.x + 5 + heading_surf.get_width() + 2
        cy = heading_box.y + 5
        if cursor_visible:
            pygame.draw.line(screen, (0, 0, 0), (cx, cy), (cx, cy + heading_box.height - 10), 1)
    elif active_duration:
        cx = duration_box.x + 5 + duration_surf.get_width() + 2
        cy = duration_box.y + 5
        if cursor_visible:
            pygame.draw.line(screen, (0, 0, 0), (cx, cy), (cx, cy + duration_box.height - 10), 1)

    # Arrow buttons
    pygame.draw.rect(screen, (220, 220, 220), heading_up)
    pygame.draw.rect(screen, (0, 0, 0), heading_up, 1)
    pygame.draw.rect(screen, (220, 220, 220), heading_down)
    pygame.draw.rect(screen, (0, 0, 0), heading_down, 1)
    pygame.draw.rect(screen, (220, 220, 220), duration_up)
    pygame.draw.rect(screen, (0, 0, 0), duration_up, 1)
    pygame.draw.rect(screen, (220, 220, 220), duration_down)
    pygame.draw.rect(screen, (0, 0, 0), duration_down, 1)

    up_surf = font.render("▲", True, (0, 0, 0))
    down_surf = font.render("▼", True, (0, 0, 0))
    screen.blit(up_surf, up_surf.get_rect(center=heading_up.center))
    screen.blit(down_surf, down_surf.get_rect(center=heading_down.center))
    screen.blit(up_surf, up_surf.get_rect(center=duration_up.center))
    screen.blit(down_surf, down_surf.get_rect(center=duration_down.center))

    # BUTTONS
    pygame.draw.rect(screen, (200, 200, 200), send_button)
    pygame.draw.rect(screen, (0, 0, 0), send_button, 2)
    screen.blit(font.render("SEND", True, (0, 0, 0)),
                (send_button.x + 15, send_button.y + 8))

    pygame.draw.rect(screen, (255, 180, 180), stop_button)
    pygame.draw.rect(screen, (0, 0, 0), stop_button, 2)
    screen.blit(font.render("STOP", True, (0, 0, 0)),
                (stop_button.x + 15, stop_button.y + 8))

    # EVENTS
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

        if event.type == pygame.MOUSEBUTTONDOWN:
            mx, my = event.pos

            # input focus
            if heading_box.collidepoint(event.pos):
                active_heading = True
                active_duration = False
            elif duration_box.collidepoint(event.pos):
                active_heading = False
                active_duration = True
            else:
                active_heading = active_duration = False

            # arrow buttons
            if heading_up.collidepoint(event.pos):
                try:
                    val = float(heading_text) if heading_text else 0.0
                except:
                    val = 0.0
                val += 1.0
                heading_text = f"{val:.1f}"
                command_heading = angle_wrap_deg(val)
            if heading_down.collidepoint(event.pos):
                try:
                    val = float(heading_text) if heading_text else 0.0
                except:
                    val = 0.0
                val -= 1.0
                heading_text = f"{val:.1f}"
                command_heading = angle_wrap_deg(val)

            if duration_up.collidepoint(event.pos):
                try:
                    val = float(duration_text) if duration_text else 0.0
                except:
                    val = 0.0
                val += 0.5
                duration_text = f"{val:.1f}"
            if duration_down.collidepoint(event.pos):
                try:
                    val = float(duration_text) if duration_text else 0.0
                except:
                    val = 0.0
                val = max(0.0, val - 0.5)
                duration_text = f"{val:.1f}"

            # buttons
            if send_button.collidepoint(event.pos):
                try:
                    deg = float(heading_text)
                    sec = float(duration_text)
                    send_heading(deg, int(sec * 1000))
                    command_heading = angle_wrap_deg(deg)
                    current_cmd_heading = deg
                    current_cmd_duration = sec
                    heading_samples = []
                    avg_heading = None
                    print(f"[CMD] HEAD {deg} for {sec}s")
                except:
                    print("[ERR] Invalid heading/duration")

            if stop_button.collidepoint(event.pos):
                send_stop()
                print("[CMD] STOP")

            # compass click
            dist = math.hypot(mx - COMPASS_CX, my - COMPASS_CY)
            if dist <= COMPASS_R:
                compass_dragging = True
                command_heading = compass_angle_from_mouse(mx, my)
                heading_text = f"{command_heading:.1f}"
                current_cmd_heading = command_heading

        if event.type == pygame.MOUSEBUTTONUP:
            compass_dragging = False

        if event.type == pygame.MOUSEMOTION and compass_dragging:
            mx, my = event.pos
            command_heading = compass_angle_from_mouse(mx, my)
            heading_text = f"{command_heading:.1f}"
            current_cmd_heading = command_heading

        if event.type == pygame.KEYDOWN:
            if event.key == pygame.K_ESCAPE:
                running = False

            if active_heading:
                if event.key == pygame.K_BACKSPACE:
                    heading_text = heading_text[:-1]
                elif event.key == pygame.K_RETURN:
                    pass
                else:
                    heading_text += event.unicode

            elif active_duration:
                if event.key == pygame.K_BACKSPACE:
                    duration_text = duration_text[:-1]
                elif event.key == pygame.K_RETURN:
                    pass
                else:
                    duration_text += event.unicode

    pygame.display.flip()

# Cleanup
stop_logging_session()
pygame.quit()


#include <WiFi.h>
#include <WiFiUdp.h>
#include <WebServer.h>
#include <Wire.h>
#include <TinyGPSPlus.h>
#include <Adafruit_Sensor.h>
#include <Adafruit_BNO055.h>

// ==========================
// NETWORK
// ==========================
WebServer server(80);
WiFiUDP udp;
#define UDP_PORT 4210

// ==========================
// MOTOR PINS
// ==========================
#define A_IN1 19
#define A_IN2 18
#define A_ENA 17

#define B_IN1 25
#define B_IN2 26
#define B_ENB 27

int dutyA = 0, dirA = 0;
int dutyB = 0, dirB = 0;

unsigned long lastPacketTime = 0;
const unsigned long FAILSAFE_TIMEOUT = 300;

// ==========================
// GPS
// ==========================
TinyGPSPlus gps;
HardwareSerial GPSserial(2);  // UART2 on pins 4/15

double gpsLat = 0, gpsLon = 0;
bool gpsFix = false;

// ==========================
// IMU / HEADING
// ==========================
Adafruit_BNO055 bno = Adafruit_BNO055(55, 0x28);

float Kp = 2.0, Ki = 0.0, Kd = 0.5;
float pidIntegral = 0, lastError = 0;

bool headingMode = false;
float targetHeading = 0;
unsigned long headingEndTime = 0;

// ==========================
// MOTOR CONTROL
// ==========================
void setMotor(int in1, int in2, int ena, int speed, int dir) {
  speed = constrain(speed, 0, 255);

  if (dir == 1) { digitalWrite(in1, HIGH); digitalWrite(in2, LOW); }
  else if (dir == -1) { digitalWrite(in1, LOW); digitalWrite(in2, HIGH); }
  else { digitalWrite(in1, LOW); digitalWrite(in2, LOW); }

  analogWrite(ena, speed);
}

void updateMotors() {
  setMotor(A_IN1, A_IN2, A_ENA, dutyA, dirA);
  setMotor(B_IN1, B_IN2, B_ENB, dutyB, dirB);
}

// ==========================
// HEADING HELPERS
// ==========================
float headingError(float target, float current) {
  float e = target - current;
  while (e > 180) e -= 360;
  while (e < -180) e += 360;
  return e;
}

// ==========================
// HEADING PID WITH FULL-POWER MODE
// ==========================
void driveHeadingPID() {
  sensors_event_t event;
  bno.getEvent(&event, Adafruit_BNO055::VECTOR_EULER);
  float heading = event.orientation.x;

  float error = headingError(targetHeading, heading);

  pidIntegral += error;
  float derivative = error - lastError;
  lastError = error;

  float correction = Kp * error + Ki * pidIntegral + Kd * derivative;

  int left, right;

  // BIG ERROR: > 30°
  if (fabs(error) > 30) {
    int baseSpeed = 150;
    left  = constrain(baseSpeed - correction, -255, 255);
    right = constrain(baseSpeed + correction, -255, 255);
  }
  // MEDIUM ERROR: 5°–30°
  else if (fabs(error) > 5) {
    int baseSpeed = 200;
    left  = constrain(baseSpeed - correction, -255, 255);
    right = constrain(baseSpeed + correction, -255, 255);
  }
  // SMALL ERROR: < 5° — FULL POWER + TRIM
  else {
    int full = 255;
    int trim = constrain((int)correction, -80, 80);

    left  = full - trim;
    right = full + trim;

    left  = constrain(left, 0, 255);
    right = constrain(right, 0, 255);
  }

  dutyA = abs(left);
  dirA  = (left >= 0) ? 1 : -1;

  dutyB = abs(right);
  dirB  = (right >= 0) ? 1 : -1;

  updateMotors();
}

// ==========================
// HTTP STATUS
// ==========================
void handleStatus() {
  sensors_event_t event;
  bno.getEvent(&event, Adafruit_BNO055::VECTOR_EULER);
  float heading = event.orientation.x;

  String json = "{";

  json += "\"heading\":" + String(heading) + ",";
  json += "\"lat\":" + String(gpsLat, 6) + ",";
  json += "\"lon\":" + String(gpsLon, 6) + ",";
  json += "\"fix\":" + String(gpsFix ? 1 : 0) + ",";

  json += "\"A_speed\":" + String(dutyA) + ",";
  json += "\"A_dir\":" + String(dirA) + ",";
  json += "\"B_speed\":" + String(dutyB) + ",";
  json += "\"B_dir\":" + String(dirB) + ",";

  json += "\"active\":" + String(headingMode ? 1 : 0) + ",";
  json += "\"RSSI\":" + String(WiFi.RSSI());

  json += "}";

  server.send(200, "application/json", json);
}

// ==========================
// SETUP
// ==========================
void setup() {
  Serial.begin(115200);

  pinMode(A_IN1, OUTPUT);
  pinMode(A_IN2, OUTPUT);
  pinMode(A_ENA, OUTPUT);
  pinMode(B_IN1, OUTPUT);
  pinMode(B_IN2, OUTPUT);
  pinMode(B_ENB, OUTPUT);

  // GPS
  GPSserial.begin(9600, SERIAL_8N1, 4, 15);

  // IMU
  Wire.begin(21, 22);
  delay(1000);
  bno.begin();
  delay(1000);
  bno.setExtCrystalUse(true);
  bno.setMode(OPERATION_MODE_NDOF);

  // WiFi AP
  WiFi.mode(WIFI_AP);
  WiFi.softAP("Rover", "12345678");

  udp.begin(UDP_PORT);

  server.on("/status", handleStatus);
  server.begin();

  lastPacketTime = millis();
}

// ==========================
// LOOP
// ==========================
void loop() {

  // GPS read
  while (GPSserial.available()) gps.encode(GPSserial.read());
  gpsFix = gps.location.isValid();
  if (gpsFix) {
    gpsLat = gps.location.lat();
    gpsLon = gps.location.lng();
  }

  server.handleClient();

  // HEADING MODE
  if (headingMode) {
    if (millis() < headingEndTime) {
      driveHeadingPID();
    } else {
      headingMode = false;
      dutyA = dutyB = 0;
      dirA = dirB = 0;
      updateMotors();
    }
  }

  // UDP commands
  int packetSize = udp.parsePacket();
  if (packetSize) {
    char buf[64];
    int len = udp.read(buf, sizeof(buf) - 1);
    buf[len] = '\0';

    // Heading command
    if (strncmp(buf, "HEAD", 4) == 0) {
      float deg;
      int dur;
      if (sscanf(buf, "HEAD,%f,%d", &deg, &dur) == 2) {
        headingMode = true;
        targetHeading = deg;
        headingEndTime = millis() + dur;
        pidIntegral = 0;
        lastError = 0;
      }
    }
    // Manual joystick
    else {
      int sA, dA, sB, dB;
      if (sscanf(buf, "%d,%d,%d,%d", &sA, &dA, &sB, &dB) == 4) {
        if (!headingMode) {
          dutyA = sA; dirA = dA;
          dutyB = sB; dirB = dB;
          updateMotors();
        }
      }
    }

    lastPacketTime = millis();
  }

  // FAILSAFE
  if (!headingMode && millis() - lastPacketTime > FAILSAFE_TIMEOUT) {
    dutyA = dutyB = 0;
    dirA = dirB = 0;
    updateMotors();
  }
}
#include <WiFi.h>
#include <WiFiUdp.h>
#include <WebServer.h>
#include <Wire.h>
#include <TinyGPSPlus.h>
#include <Adafruit_Sensor.h>
#include <Adafruit_BNO055.h>

// ==========================
// NETWORK
// ==========================
WebServer server(80);
WiFiUDP udp;
#define UDP_PORT 4210

// ==========================
// MOTOR PINS
// ==========================
#define A_IN1 19
#define A_IN2 18
#define A_ENA 17

#define B_IN1 25
#define B_IN2 26
#define B_ENB 27

int dutyA = 0, dirA = 0;
int dutyB = 0, dirB = 0;

unsigned long lastPacketTime = 0;
const unsigned long FAILSAFE_TIMEOUT = 300;

// ==========================
// GPS
// ==========================
TinyGPSPlus gps;
HardwareSerial GPSserial(2);  // UART2 on pins 4/15

double gpsLat = 0, gpsLon = 0;
bool gpsFix = false;

// ==========================
// IMU / HEADING
// ==========================
Adafruit_BNO055 bno = Adafruit_BNO055(55, 0x28);

float Kp = 2.0, Ki = 0.0, Kd = 0.5;
float pidIntegral = 0, lastError = 0;

bool headingMode = false;
float targetHeading = 0;
unsigned long headingEndTime = 0;

// ==========================
// MOTOR CONTROL
// ==========================
void setMotor(int in1, int in2, int ena, int speed, int dir) {
  speed = constrain(speed, 0, 255);

  if (dir == 1) { digitalWrite(in1, HIGH); digitalWrite(in2, LOW); }
  else if (dir == -1) { digitalWrite(in1, LOW); digitalWrite(in2, HIGH); }
  else { digitalWrite(in1, LOW); digitalWrite(in2, LOW); }

  analogWrite(ena, speed);
}

void updateMotors() {
  setMotor(A_IN1, A_IN2, A_ENA, dutyA, dirA);
  setMotor(B_IN1, B_IN2, B_ENB, dutyB, dirB);
}

// ==========================
// HEADING HELPERS
// ==========================
float headingError(float target, float current) {
  float e = target - current;
  while (e > 180) e -= 360;
  while (e < -180) e += 360;
  return e;
}

// ==========================
// HEADING PID WITH FULL-POWER MODE
// ==========================
void driveHeadingPID() {
  sensors_event_t event;
  bno.getEvent(&event, Adafruit_BNO055::VECTOR_EULER);
  float heading = event.orientation.x;

  float error = headingError(targetHeading, heading);

  pidIntegral += error;
  float derivative = error - lastError;
  lastError = error;

  float correction = Kp * error + Ki * pidIntegral + Kd * derivative;

  int left, right;

  // BIG ERROR: > 30°
  if (fabs(error) > 30) {
    int baseSpeed = 150;
    left  = constrain(baseSpeed - correction, -255, 255);
    right = constrain(baseSpeed + correction, -255, 255);
  }
  // MEDIUM ERROR: 5°–30°
  else if (fabs(error) > 5) {
    int baseSpeed = 200;
    left  = constrain(baseSpeed - correction, -255, 255);
    right = constrain(baseSpeed + correction, -255, 255);
  }
  // SMALL ERROR: < 5° — FULL POWER + TRIM
  else {
    int full = 255;
    int trim = constrain((int)correction, -80, 80);

    left  = full - trim;
    right = full + trim;

    left  = constrain(left, 0, 255);
    right = constrain(right, 0, 255);
  }

  dutyA = abs(left);
  dirA  = (left >= 0) ? 1 : -1;

  dutyB = abs(right);
  dirB  = (right >= 0) ? 1 : -1;

  updateMotors();
}

// ==========================
// HTTP STATUS
// ==========================
void handleStatus() {
  sensors_event_t event;
  bno.getEvent(&event, Adafruit_BNO055::VECTOR_EULER);
  float heading = event.orientation.x;

  String json = "{";

  json += "\"heading\":" + String(heading) + ",";
  json += "\"lat\":" + String(gpsLat, 6) + ",";
  json += "\"lon\":" + String(gpsLon, 6) + ",";
  json += "\"fix\":" + String(gpsFix ? 1 : 0) + ",";

  json += "\"A_speed\":" + String(dutyA) + ",";
  json += "\"A_dir\":" + String(dirA) + ",";
  json += "\"B_speed\":" + String(dutyB) + ",";
  json += "\"B_dir\":" + String(dirB) + ",";

  json += "\"active\":" + String(headingMode ? 1 : 0) + ",";
  json += "\"RSSI\":" + String(WiFi.RSSI());

  json += "}";

  server.send(200, "application/json", json);
}

// ==========================
// SETUP
// ==========================
void setup() {
  Serial.begin(115200);

  pinMode(A_IN1, OUTPUT);
  pinMode(A_IN2, OUTPUT);
  pinMode(A_ENA, OUTPUT);
  pinMode(B_IN1, OUTPUT);
  pinMode(B_IN2, OUTPUT);
  pinMode(B_ENB, OUTPUT);

  // GPS
  GPSserial.begin(9600, SERIAL_8N1, 4, 15);

  // IMU
  Wire.begin(21, 22);
  delay(1000);
  bno.begin();
  delay(1000);
  bno.setExtCrystalUse(true);
  bno.setMode(OPERATION_MODE_NDOF);

  // WiFi AP
  WiFi.mode(WIFI_AP);
  WiFi.softAP("Rover", "12345678");

  udp.begin(UDP_PORT);

  server.on("/status", handleStatus);
  server.begin();

  lastPacketTime = millis();
}

// ==========================
// LOOP
// ==========================
void loop() {

  // GPS read
  while (GPSserial.available()) gps.encode(GPSserial.read());
  gpsFix = gps.location.isValid();
  if (gpsFix) {
    gpsLat = gps.location.lat();
    gpsLon = gps.location.lng();
  }

  server.handleClient();

  // HEADING MODE
  if (headingMode) {
    if (millis() < headingEndTime) {
      driveHeadingPID();
    } else {
      headingMode = false;
      dutyA = dutyB = 0;
      dirA = dirB = 0;
      updateMotors();
    }
  }

  // UDP commands
  int packetSize = udp.parsePacket();
  if (packetSize) {
    char buf[64];
    int len = udp.read(buf, sizeof(buf) - 1);
    buf[len] = '\0';

    // Heading command
    if (strncmp(buf, "HEAD", 4) == 0) {
      float deg;
      int dur;
      if (sscanf(buf, "HEAD,%f,%d", &deg, &dur) == 2) {
        headingMode = true;
        targetHeading = deg;
        headingEndTime = millis() + dur;
        pidIntegral = 0;
        lastError = 0;
      }
    }
    // Manual joystick
    else {
      int sA, dA, sB, dB;
      if (sscanf(buf, "%d,%d,%d,%d", &sA, &dA, &sB, &dB) == 4) {
        if (!headingMode) {
          dutyA = sA; dirA = dA;
          dutyB = sB; dirB = dB;
          updateMotors();
        }
      }
    }

    lastPacketTime = millis();
  }

  // FAILSAFE
  if (!headingMode && millis() - lastPacketTime > FAILSAFE_TIMEOUT) {
    dutyA = dutyB = 0;
    dirA = dirB = 0;
    updateMotors();
  }
}
